In [1]:
from hossam import load_data
from pandas import DataFrame, melt
from matplotlib import pyplot as plt
from matplotlib import font_manager as fm
import seaborn as sb
from math import sqrt
# 가정 확인을 위한 라이브러리
from scipy.stats import t, normaltest, bartlett, levene
# 분산분석을 위한 라이브러리
from pingouin import anova
from pingouin import welch_anova
# 사후검정을 위한 라이브러리
from pingouin import pairwise_tukey, pairwise_gameshowell

In [2]:
my_dpi = 200 # 이미지 선명도(100~300)
fpath = "./NotoSansKR-Regular.ttf" # 한글을 지원하는 폰트 파일의 경로
fm.fontManager.addfont(fpath) # 폰트의 글꼴을 시스템에 등록함
fprop = fm.FontProperties(fname=fpath) # 폰트의 속성을 읽어옴
fname = fprop.get_name() # 읽어온 속성에서 폰트 이름 추출
plt.rcParams['font.family'] = fname # 그래프에 한글 폰트 적용
plt.rcParams['font.size'] = 6 # 기본 폰트 크기
plt.rcParams['axes.unicode_minus'] = False # 그래프에 마이너스 깨짐 방지 

📘 #02. 예제1 - 측정자에 따른 태아의 머리 둘레 측정 비교<br>
다음의 데이터는 3명의 태아를 대상으로 3명의 관측자가 측정한 자료이다.<br>
측정자에 따라 태아의 머리 둘레가 다르게 타나나는지 확인하라.<br>

In [3]:
origin = load_data('head_size')
print("\n===== 데이터 크기 확인 =====")
print(f"데이터셋 크기: {origin.shape}")
print(f"열 개수: {origin.shape[1]}")
print(f"행 개수: {origin.shape[0]}")
print("\n===== 타입확인 =====")
print(origin.info())
origin.head()

[data] https://data.hossam.kr/data/lab10_/head_size.xlsx
[desc] 3명의 태아를 대상으로 3명의 관측자가 측정한 태아의 머리 둘레 자료 (출처: 방송통신대학교 통계학 개론)
[!] Cannot read metadata

===== 데이터 크기 확인 =====
데이터셋 크기: (60, 4)
열 개수: 4
행 개수: 60

===== 타입확인 =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   측정자번호   60 non-null     int64  
 1   1번태아    60 non-null     float64
 2   2번태아    60 non-null     float64
 3   3번태아    60 non-null     float64
dtypes: float64(3), int64(1)
memory usage: 2.0 KB
None


,측정자번호,1번태아,2번태아,3번태아
0,1,14.9,19.7,13.0
1,1,14.4,20.7,13.5
2,1,14.4,19.9,13.2
3,1,15.1,20.2,12.8
4,1,15.4,19.4,13.8


📝 [2] 데이터 전처리<br>
이원 분산분석을 수행하기 적합한 형태는 melt 처리된 구조이다.<br>

In [5]:
df = melt(origin, id_vars='측정자번호',
value_vars=['1번태아', '2번태아', '3번태아'],
var_name='태아번호', value_name='머리둘레')
df.head()
df

,측정자번호,태아번호,머리둘레
0,1,1번태아,14.9
1,1,1번태아,14.4
2,1,1번태아,14.4
3,1,1번태아,15.1
4,1,1번태아,15.4
...,...,...,...
175,3,3번태아,11.0
176,3,3번태아,10.2
177,3,3번태아,10.1
178,3,3번태아,10.2


📝 [3] 데이터 분포 시각화

1) 측정자별 태아에 따른 머리 둘레 비교